# Tutorial 04 — Optimization principles

SoftMobility wraps the soft-body rollout in a JAX-friendly objective so
that **design parameters** can be optimised by gradient descent.

This tutorial walks through the workflow on a small, fully-determined
example: an **asymmetric dumbbell** sinking under gravity. Two spheres
are placed on either side of the origin. The radius of the left sphere
is fixed at `r₀ = 0.5`; we optimise the radius `r₁` of the right one so
that the dumbbell falls **vertically** (zero lateral drift) instead of
tipping under unequal drag.

We use:

* `softmobility.SoftBody` for the body,
* `softmobility.FlowBodyRollout` for the trajectory,
* `softmobility.FlowBodyOptimizer` to wrap an Optax optimiser around a
  scalar objective.


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import plotly.graph_objects as go

from softmobility import (
    FlowBodyOptimizer,
    FlowBodyRollout,
    SoftBody,
    gravity_field,
    no_flow,
)

np.set_printoptions(precision=3, suppress=True)


## Step 1 — Body

The right-sphere radius `r₁` is the only design parameter. Each sphere
carries a gravity-coupled body force `m·g` with `m = (4/3)π r³`:


In [2]:
yaml_body = """
design_names: [r1]
input_names: [gravity]
defaults:
  r1: 0.9
spheres:
  - radius: 0.5
    position: [-1, 0, 0]
    force: [0.523599 * gravity0, 0.523599 * gravity1, 0.523599 * gravity2]
  - radius: r1
    position: [ 1, 0, 0]
    force: [4.18879 * r1**3 * gravity0, 4.18879 * r1**3 * gravity1, 4.18879 * r1**3 * gravity2]
"""

body = SoftBody(yaml_body, verbose=False)
print(repr(body))


SPHERE ASSEMBLY
  2 spheres
  0 degrees of freedom
  1 design parameters
  3 input parameters

Default values
  degrees of freedom dof: [] = []
  design parameters param: ['r1'] = [0.9]
  input parameters param: ['gravity0', 'gravity1', 'gravity2']

SPHERE 0
  radius: 0.5
  position: [-1.  0.  0.]
  orientation: [0. 0. 0.]
  C_H:
[[0.524 0.    0.   ]
 [0.    0.524 0.   ]
 [0.    0.    0.524]
 [0.    0.    0.   ]
 [0.    0.    0.   ]
 [0.    0.    0.   ]]
  C_K:
[]

SPHERE 1
  radius: 0.8999999761581421
  position: [1. 0. 0.]
  orientation: [0. 0. 0.]
  C_H:
[[3.054 0.    0.   ]
 [0.    3.054 0.   ]
 [0.    0.    3.054]
 [0.    0.    0.   ]
 [0.    0.    0.   ]
 [0.    0.    0.   ]]
  C_K:
[]



## Step 2 — Rollout

Gravity along `-ẑ`, no flow. The body starts horizontal (along the lab
*x*-axis). With unequal radii the heavier sphere experiences more
gravitational force but its drag scales differently (mass ∝ r³, drag ∝
r), so the body tilts and drifts laterally.


In [3]:
rollout = FlowBodyRollout(
    soft_body=body,
    flow=no_flow(),
    input_map={"gravity": gravity_field(g=9.81)},
)

dt = 0.005
n_steps = 400


## Step 3 — Define the scalar objective

We want the dumbbell to fall vertically: minimise the **lateral drift**
of the body-reference point at the end of the rollout.

The objective signature must be `(rollout, design) -> scalar` so JAX can
trace and differentiate it.


In [ ]:
def lateral_drift_squared(rollout, design):
    positions, _, _ = rollout.rollout(
        dt=dt,
        n_steps=n_steps,
        design=design,
    )
    final = positions[-1]
    return final[0] ** 2 + final[1] ** 2  # ignore the z (vertical) component


# Sanity check: gradient at the symmetric point r1 = 0.5 should be ~0.
g0 = jax.grad(lambda d: lateral_drift_squared(rollout, d))(jnp.array([0.5]))
g1 = jax.grad(lambda d: lateral_drift_squared(rollout, d))(jnp.array([0.9]))
print(f"∂/∂r1 of drift² at r1=0.5 : {float(g0[0]):.4e}")
print(f"∂/∂r1 of drift² at r1=0.9 : {float(g1[0]):.4e}")


## Step 4 — Run the optimiser

Adam, learning rate 0.05, 60 steps. `r₁` is clipped to `[0.1, 2]`.


In [5]:
optimizer = FlowBodyOptimizer(
    rollout,
    lateral_drift_squared,
    optax.adam(learning_rate=0.05),
)

init_design = jnp.array([0.9])
optimal_design = optimizer.run(
    init_design=init_design,
    n_steps=60,
    print_every=10,
    clip_min=0.1,
    clip_max=2.0,
    maximize=False,
)
print(f"\nfinal r1 = {float(optimal_design[0]):.4f}  (expected ≈ 0.5 for symmetric dumbbell)")


step    0  objective=-0.0144  |grad|=0.0957  design0=0.85
step   10  objective=-0.0000  |grad|=0.0002  design0=0.47


step   20  objective=-0.0002  |grad|=0.0025  design0=0.34
step   30  objective=-0.0003  |grad|=0.0033  design0=0.32


step   40  objective=-0.0002  |grad|=0.0025  design0=0.35
step   50  objective=-0.0001  |grad|=0.0016  design0=0.40


step   59  objective=-0.0000  |grad|=0.0011  design0=0.43

final r1 = 0.4691  (expected ≈ 0.5 for symmetric dumbbell)


## Step 5 — Compare trajectories

Plot the centre-of-mass trajectory before vs after optimisation. The
initial asymmetric body drifts sideways while the optimised body falls
vertically.


In [6]:
def trajectory(design):
    positions, _, _ = rollout.rollout(
        dt=dt,
        n_steps=n_steps,
        design=design,
    )
    return np.asarray(positions)


traj_init = trajectory(init_design)
traj_opt = trajectory(optimal_design)

fig = go.Figure()
fig.add_trace(go.Scatter(x=traj_init[:, 0], y=traj_init[:, 2],
                         mode="lines",
                         name=f"initial r₁ = {float(init_design[0]):.2f}",
                         line=dict(color="#888", width=2, dash="dash")))
fig.add_trace(go.Scatter(x=traj_opt[:, 0], y=traj_opt[:, 2],
                         mode="lines",
                         name=f"optimised r₁ = {float(optimal_design[0]):.3f}",
                         line=dict(color="#1f77b4", width=3)))
fig.update_layout(
    title="Centre-of-mass trajectory before / after optimisation",
    xaxis_title="x  (lateral drift)",
    yaxis_title="z  (vertical fall)",
    yaxis=dict(scaleanchor="x", scaleratio=1),
    width=700, height=500,
    plot_bgcolor="white",
)
fig.show()


## Notes

* Here the body is **rigid** (no DOFs); only the design `r₁` changes.
  The same workflow applies when the body has DOFs: the rollout
  integrates them, the optimiser only sees the scalar objective.
* For higher-dimensional design spaces (swimmer geometry, fiber-element
  radii) the same `FlowBodyOptimizer.run(...)` call works directly — see
  tutorial **22 — three-sphere swimmer** for a 6-parameter case.
* The optimiser uses `jax.value_and_grad`, so the objective must be
  built **purely** from JAX operations on the rollout output.
